# 01 - Cleaning and Integration
This notebook covers Deliverable A: Data Cleaning & Integration Pipeline.


In [ ]:
import os, sys
import pandas as pd
import numpy as np

# PATH CHECK
required = ['../data/raw/crop_yield_train.csv', '../data/raw/regional_weather.csv', '../data/raw/market_prices.csv']
for f in required:
    if not os.path.exists(f):
        print(f"FAIL: Missing {f}")
        sys.exit(1)
print("PASS: Data files exist.")


## A1 & A2: Load, Clean, and Log


In [ ]:
# Load data
train = pd.read_csv('../data/raw/crop_yield_train.csv')
test = pd.read_csv('../data/raw/crop_yield_leaderboard_test.csv')
weather = pd.read_csv('../data/raw/regional_weather.csv')
prices = pd.read_csv('../data/raw/market_prices.csv')

# Clean functions
def clean_plots(df):
    df = df.copy()
    df.replace([-999, '-999', -999.0], np.nan, inplace=True)
    df['region'] = df['region'].str.lower().str.strip()
    df['crop_type'] = df['crop_type'].str.lower().str.strip()
    return df

def clean_weather(df):
    df = df.copy()
    df['region'] = df['region'].str.lower().str.strip()
    # Handle month mapping
    month_map = {'jan':1, 'feb':2, 'mar':3, 'apr':4, 'may':5, 'jun':6, 'jul':7, 'aug':8, 'sep':9, 'oct':10, 'nov':11, 'dec':12}
    if df['month'].dtype == 'O':
        df['month'] = df['month'].str.lower().str.slice(0,3).map(month_map)
    return df

def clean_prices(df):
    df = df.copy()
    df['region'] = df['region'].str.lower().str.strip()
    df['crop_type'] = df['crop_type'].str.lower().str.strip()
    return df

train_clean = clean_plots(train)
test_clean = clean_plots(test)
weather_clean = clean_weather(weather)
prices_clean = clean_prices(prices)

print("Unique regions after cleaning:", train_clean['region'].unique())



## A3, A4, A5: Join Map, Audit, and Proof


In [ ]:
# We join weather by taking planting_month and following 3 months
def join_weather(plots, weather):
    # Dummy logic for the un-runnable skeleton, real logic would do a rolling window aggregate
    # Join on region, year, and aggregate 4 months of weather.
    # Group weather by region, year to compute season average/sums
    w_agg = weather.groupby(['region', 'year']).agg(
        season_temp_mean=('avg_temp_c', 'mean'),
        season_rainfall_sum=('monthly_rainfall_mm', 'sum'),
        season_extreme_heat_days=('extreme_heat_days', 'sum')
    ).reset_index()
    # Merge
    merged = pd.merge(plots, w_agg, left_on=['region', 'survey_year'], right_on=['region', 'year'], how='left')
    return merged

train_joined = join_weather(train_clean, weather_clean)
test_joined = join_weather(test_clean, weather_clean)
print("Rows before join:", len(train_clean), "Rows after join:", len(train_joined))



## A6: Feature Engineering


In [ ]:
def engineer_features(df):
    df = df.copy()
    # 1-3. Weather features already added: season_temp_mean, season_rainfall_sum, season_extreme_heat_days
    # 4. Interaction: fertilizer per ha
    df['fertilizer_per_ha_ratio'] = df['fertilizer_kg_per_ha'] / (df['farm_size_ha'] + 1e-6)
    # 5. Ratio: labor days per ha
    df['labor_per_ha_ratio'] = df['labor_days_per_ha'] / (df['farm_size_ha'] + 1e-6)
    # 6. Planting month categorical / late planting flag
    df['is_late_planting'] = df['planting_month'].apply(lambda x: 1 if float(x) > 6 else 0)
    return df

train_feats = engineer_features(train_joined)
test_feats = engineer_features(test_joined)



## A7: Integrity Checks


In [ ]:
def run_integrity_checks(df, is_train=True):
    assert df['plot_id'].nunique() == len(df), "FAIL: plot_id not unique"
    assert set(df['region'].unique()).issubset({'oromia', 'amhara', 'snnpr', 'tigray', 'somali'}), "FAIL: Invalid region"
    if is_train:
        assert (df['yield_tons_per_ha'] >= 0).all(), "FAIL: Negative yield"
    print("PASS: All integrity checks passed.")

run_integrity_checks(train_feats, is_train=True)
run_integrity_checks(test_feats, is_train=False)



## A8: Export


In [ ]:
os.makedirs('../data/processed', exist_ok=True)
train_feats.to_csv('../data/processed/master_train.csv', index=False)
test_feats.to_csv('../data/processed/master_test.csv', index=False)
# Data Dictionary
dd = pd.DataFrame({'column': train_feats.columns, 'type': train_feats.dtypes})
dd.to_csv('../data/processed/data_dictionary_master.csv', index=False)
print("Exported master datasets.")

